In [4]:
import os
import sys
from pathlib import Path

from langchain_openai import ChatOpenAI
from rag_system.vector_store import VectorStoreManager
from rag_system.orchestrator import Orchestrator

from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
load_dotenv()

vsm = VectorStoreManager(data_dir="data/")

faiss_path = project_root / "faiss_indices"
if faiss_path.exists():
    vsm.load(str(faiss_path))
    print("Loaded indices from disk")
else:
    vsm.initialize()
    vsm.save(str(faiss_path))
    print("Built and saved indices")
    
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

orchestrator = Orchestrator(llm, vsm)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1431.15it/s]


Loaded indices from disk


In [5]:
test_queries = [
    "What's the process for deploying a new microservice and what compliance checks are needed?",
    "How do I troubleshoot API performance issues while following our security policies?",
    "What business approvals are required for implementing a new data processing workflow?"
]

# Simulated per-scenario feedback (keyed by scenario index)
simulated_feedback = {
    1: {"technical": 0.9, "compliance": 0.8},
    2: {"technical": 0.7, "compliance": 1.0},
    3: {"business": 0.95},
}

print("Trust scores before feedback:")
print(orchestrator.feedback.trust_scores())

for i, query in enumerate(test_queries, 1):
    print("=" * 80)
    print(f"Scenario {i}: {query}")
    print("=" * 80)

    result = orchestrator.query(query)

    # Classification
    domains = [d.value for d in result.classification.domains]
    print(f"\nDomains routed to: {domains}")
    print(f"(query words = {len(query.split())}, domains = {len(domains)})")

    print("\nSub-queries:")
    for domain, sub_query in result.classification.sub_queries.items():
        print(f"  [{domain}] {sub_query}")

    # Per-agent answers + retrieved chunks with similarity scores
    print("\nPer-agent answers:")
    for ar in result.agent_responses:
        scores = [f"{c.similarity_score:.4f}" for c in ar.retrieved_chunks]
        print(f"\n  [{ar.domain.value.upper()}]  retrieved = {len(ar.retrieved_chunks)}  "
              f"similarity_scores = {scores}")
        print(f"\nDomain Agent Answer: {ar.answer}")

    # Citations
    print(f"\nCitations ({len(result.citations)}):")
    for citation in result.citations:
        print(citation)

    print("\nFinal synthesized answer:")
    print(result.answer)
    
    # Recording simulated feedback
    if i in simulated_feedback:
        orchestrator.record_feedback(simulated_feedback[i])
        print(f"\nFeedback recorded: {simulated_feedback[i]}")
        print(f"Updated trust scores: {orchestrator.feedback.trust_scores()}")
        
print("=" * 80)
print("Final trust scores after all feedback:")
print(orchestrator.feedback.trust_scores())

Trust scores before feedback:
{'technical': 1.0, 'business': 1.0, 'compliance': 1.0}
Scenario 1: What's the process for deploying a new microservice and what compliance checks are needed?

Domains routed to: ['technical', 'compliance']
(query words = 14, domains = 2)

Sub-queries:
  [technical] What are the detailed steps and best practices for deploying a new microservice within our infrastructure, including CI/CD pipelines and monitoring setup?
  [compliance] What compliance checks and security policies must be verified before and after deploying a new microservice to ensure regulatory and internal standards are met?

Per-agent answers:

  [TECHNICAL]  retrieved = 6  similarity_scores = ['0.6111', '0.9076', '0.9304', '0.9448', '0.9687', '1.0573']

Domain Agent Answer: The detailed steps and best practices for deploying a new microservice within Postbank's infrastructure, including CI/CD pipelines and monitoring setup, are as follows:

1. Development and Build Stage:
   - Developers p

Here are some questions whose answers aren't covered in any of the three PDFs:

**Cross-domain gaps:**

- "What is the process for rotating API keys?"

- "How do I set up a new development environment locally?"

- "What is the onboarding process for new engineers joining a team?"

- "How do I request a budget for a new project?"

**Technical gaps:**

- "How do I configure auto-scaling for my microservice?"

- "What is the disaster recovery procedure if the primary data center goes down?"

- "How do I set up database replication across regions?"

**Business gaps:**

- "What is the process for hiring a contractor for a project?"

- "How do I submit an expense report?"

**Compliance gaps:**

- "What are the rules for using personal devices to access company systems?"

- "What is the policy on open-source software contributions?"

In [7]:
from rag_system.utils import Domain

# Showing that a query about a new topic returns no relevant results
result_before = orchestrator.query("What are the rules for using personal devices to access company systems?")
print("Before knowledge update:")
print(result_before.answer)
print(f"Citations: {len(result_before.citations)}\n")

for citation in result_before.citations:
    print(citation)
    
# Adding new content to the compliance domain
new_content = """
Rules for Using Personal Devices (BYOD) to Access Company Systems

1. Device Registration: Employees must register their personal devices with IT before accessing company resources.
2. Security Software: Personal devices must have up-to-date antivirus software and a secure lock screen enabled.
3. Data Access: Access to sensitive data may be restricted on personal devices. Employees must use company-approved applications for accessing corporate data.
4. Network Access: Personal devices should connect to the company network via VPN when accessing internal resources
5. Compliance: Employees must comply with all company policies regarding data security and privacy when using personal devices.
"""

chunks_added = vsm.add_document(
    domain=Domain.COMPLIANCE,
    text=new_content,
    source="byod_policy.pdf",
)
print(f"Added {chunks_added} new chunks to compliance domain\n")
print(f"{'='*80}\n")

# Same query now finds relevant content
result_after = orchestrator.query("What are the rules for using personal devices to access company systems?")
print("After knowledge update:")
print(result_after.answer)
print(f"Citations: {len(result_after.citations)}")

for citation in result_after.citations:
    print(citation)

Before knowledge update:
The provided compliance manual context does not contain any specific policies or security regulations governing the use of personal devices for accessing company systems. Therefore, I do not have information on this topic based on the given context.
Citations: 5

Citation(chunk_id='compliance_000', source='data\\compliance_knowledge.pdf', domain=<Domain.COMPLIANCE: 'compliance'>, excerpt='Compliance\tand\tSecurity\tManual\nSecurity\tPolicies\nAccess\tControl\tPolicy\nAll\taccess\tto\tPostbank\tsystem')
Citation(chunk_id='compliance_014', source='data\\compliance_knowledge.pdf', domain=<Domain.COMPLIANCE: 'compliance'>, excerpt='detection.\tThe\tDPO\tdetermines\twhether\tnotiﬁcation\tto\tthe\nsupervisory\tauthority\t(within\t72\thours)\tand')
Citation(chunk_id='compliance_008', source='data\\compliance_knowledge.pdf', domain=<Domain.COMPLIANCE: 'compliance'>, excerpt='logged:\nUser\tauthentication\tevents\t(successful\tand\tfailed\tlogin\tattempts).\nData\tacces